In [7]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. CARGAR EL MODELO (Lo que faltaba) ---
# Cambia esta ruta por la ubicación real de tu archivo .keras en Drive
PATH_AL_MODELO = '/content/drive/MyDrive/mi_modelo_mobilenet.keras'
print("Cargando el modelo... por favor espera.")
model = tf.keras.models.load_model(PATH_AL_MODELO)
print("✅ Modelo cargado exitosamente.")

# --- 2. CARGAR EL CSV ---
# Cambia esta ruta por la de tu test.csv en Drive
PATH_AL_CSV = '/content/drive/MyDrive/test.csv'
df_test = pd.read_csv(PATH_AL_CSV)
print(f"✅ CSV cargado con {len(df_test)} filas.")

# --- 3. CONFIGURAR EL GENERADOR ---
# Usamos preprocess_input para que coincida con MobileNet
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=df_test,
    directory=None,                # Las rutas ya están completas en el CSV
    x_col='file_path',             
    y_col='label',                 
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',      
    shuffle=False
)

# --- 4. EJECUTAR EL TEST (PREDICCIONES) ---
print(f"Iniciando predicciones sobre {len(df_test)} imágenes...")
# Esto tardará unos minutos debido al volumen de 25,000 fotos
predictions = model.predict(test_generator)
print("✅ Predicciones completadas.")

# --- 5. MOSTRAR RESULTADOS ---
y_pred = np.argmax(predictions, axis=1)
y_true = test_generator.classes
target_names = list(test_generator.class_indices.keys())

print("\n--- REPORTE DE CLASIFICACIÓN ---")
print(classification_report(y_true, y_pred, target_names=target_names))

Cargando el modelo... por favor espera.
✅ Modelo cargado exitosamente.
✅ CSV cargado con 822 filas.
Found 822 validated image filenames belonging to 10 classes.
Iniciando predicciones sobre 822 imágenes...
26/26 ━━━━━━━━━━━━━━━━━━━━ 213s 8s/step
✅ Predicciones completadas.

--- REPORTE DE CLASIFICACIÓN ---
              precision    recall  f1-score   support

     battery       0.95      0.91      0.92        95
  biological       0.98      1.00      0.99        99
 brown-glass       0.95      0.90      0.92        60
   cardboard       0.99      0.97      0.98        89
 green-glass       0.93      0.90      0.92        63
       metal       0.81      0.87      0.84        77
       paper       0.95      0.93      0.94       105
     plastic       0.90      0.70      0.79        87
       trash       0.96      0.96      0.96        70
 white-glass       0.71      0.94      0.81        77

    accuracy                           0.91       822
   macro avg       0.91      0.91      0.9

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# 1. Calcular la matriz
cm = confusion_matrix(y_true, y_pred)
target_names = list(test_generator.class_indices.keys())

# 2. Configurar el gráfico
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=target_names, 
            yticklabels=target_names)

# 3. Añadir etiquetas para que sea fácil de leer
plt.title('Matriz de Confusión: ¿Qué está confundiendo el modelo?', fontsize=16)
plt.ylabel('Clase Real (La Verdad)', fontsize=12)
plt.xlabel('Predicción del Modelo (Lo que dijo)', fontsize=12)
plt.show()